# Week 7 · 02 Autoencoders

Everything so far needed labels. An **autoencoder** needs none: it's a network trained to output its own input, through a narrow middle layer. To rebuild 784 pixels from 8 numbers, it has to learn what digits have in common; those 8 numbers (the **code**) are a summary of the image it learned by itself. §2 proves that the simplest autoencoder, a linear one, learns exactly PCA; the nonlinear one does better. Autoencoders are also the base of `03` (the VAE) and `04` (anomaly detection).

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **autoencoder (AE)**: an encoder followed by a decoder, trained so that the output equals the input.
- **encoder**: the first half: input (784 numbers) → code (k numbers).
- **code, or latent vector $z$**: the encoder's output: k numbers that describe the input. "Latent" means hidden: nobody labels it, the network invents it.
- **bottleneck**: the narrowest layer, where the code sits. Its width k decides how much has to be thrown away.
- **decoder**: the second half: code → a reconstruction $\hat x$ of the input.
- **reconstruction error**: how far $\hat x$ is from $x$, usually the mean squared difference per pixel.
- **self-supervised**: trained without human labels, on a target made from the data itself (here: the input is its own target).

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from figures import *
import checks; checks.load(globals(), "02")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** The notebook in one picture.

In [ ]:
fig_ae_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and booklet 6.6 (PCA reconstruction) from memory | §2 shows the linear autoencoder IS this |
| E | a trained autoencoder rebuilds a test digit from 8 numbers | start from the end |
| 1 | the idea, the loss and the parameter count | what it is and what the exam asks |
| 2 | a linear autoencoder learns PCA | a proposition with its proof, and the picture |
| 3 | training it | the target is the input; what the bottleneck size does |
| 4 | the code space | what the codes look like (k = 2), and the gaps `03` fixes |
| 5 | denoising autoencoders | noisy in, clean out: a better reason to learn |
| 6 | convolutional autoencoders | the transposed convolution and its size formula |
| 7 | exercises | the block's hand-written parts |

## 0. Recap first · 10 min, closed book

**Quick check:** PCA with $k$ components: how do you get from a centred point $x_c$ to its scores, and back?

<details><summary>Answer</summary>

Scores $z = W_k^\top x_c$ (the coordinates along the top-$k$ eigenvectors, the columns of $W_k$); back: $\hat x = W_k z + \bar x$. [Week 4 · 02 §8 ↗](../week4/02_pca.ipynb)

</details>

**Quick check:** What does PCA's top eigenvector maximize, and what does that mean for the reconstruction error?

<details><summary>Answer</summary>

The variance of the projected points. Since total variance = kept + lost, maximizing what's kept minimizes the squared reconstruction error: the same direction does both. [Week 4 · 02 §5 ↗](../week4/02_pca.ipynb)

</details>

**Quick check:** Why is `nn.Sigmoid()` a good last layer for something that should output pixel values in $[0, 1]$?

<details><summary>Answer</summary>

It maps any number into $(0, 1)$. [Week 5 · 02 §4 ↗](../week5/02_neural_networks.ipynb)

</details>

**Quick check:** `nn.Linear(784, 128)`: how many parameters, and what's the shape of `.weight`?

<details><summary>Answer</summary>

$784 \cdot 128 + 128 = 100{,}480$; `.weight` is `(128, 784)`: PyTorch stores (out, in). [Week 5 · 06 §3 ↗](../week5/06_pytorch_basics.ipynb)

</details>

**Quick check:** In 01, what is the "feature extractor" of a classifier?

<details><summary>Answer</summary>

Every layer except the last: it turns the input into the features the last layer classifies. An autoencoder's encoder is a feature extractor trained without labels. [01 §1 ↗](01_transfer_learning.ipynb)

</details>

**Exercise 1 · Booklet 6.6: reconstruct with PCA.** Centre `X` `(n, d)`, covariance $S = X_c^\top X_c/(n - 1)$, `np.linalg.eigh`, sort the eigenvectors by eigenvalue (largest first), keep $k$ columns $W_k$ `(d, k)`; return $\hat X = X_c W_k W_k^\top + \bar x$, shape `(n, d)`.

**Example**: what goes in, and what should come out:
```python
X = np.array([[2.0, 1.0], [4.0, 3.0], [6.0, 5.0], [0.0, 1.0]])    # (4, 2) float
pca_reconstruct(X, 1).round(3)
# → array([[1.633, 1.503],
#          [3.891, 3.15 ],
#          [6.149, 4.796],
#          [0.327, 0.551]])
```

In [ ]:
def pca_reconstruct(X, k):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: a trained autoencoder rebuilds a test digit

`ae8.pt` in this folder is an autoencoder trained for five epochs on the 60,000 MNIST training images, by the code in §3. `figures.py`'s `mnist_tensors()` gives all of MNIST as four tensors (each image flattened to a row of 784 pixels in $[0, 1]$), and `load_ae(8)` loads the model. What it stores:

In [ ]:
X_tr, y_tr, X_te, y_te = mnist_tensors()      # X_tr: (60000, 784) float32 in [0, 1]; y_tr: (60000,), unused by the AE
model = load_ae(8)
for name, t in model.state_dict().items():
    print(name, tuple(t.shape))
print(sum(p.numel() for p in model.parameters()))
# → encoder.0.weight (128, 784)
#   encoder.0.bias (128,)
#   encoder.2.weight (8, 128)
#   encoder.2.bias (8,)
#   decoder.0.weight (128, 8)
#   decoder.0.bias (128,)
#   decoder.2.weight (784, 128)
#   decoder.2.bias (784,)
#   203800

Two halves, mirror images of each other: the **encoder** goes 784 → 128 → 8, the **decoder** 8 → 128 → 784. **A new image** from the test set, through both halves:

In [ ]:
x = X_te[:1]                                  # (1, 784): a batch of one, the first test image (a 7)
with torch.no_grad():
    z = model.encoder(x)                      # (1, 8): the code
    x_hat = model.decoder(z)                  # (1, 784): the reconstruction
print(z.numpy().round(1))
print(round(((x_hat - x) ** 2).mean().item(), 4))
# → [[-3.6 -6.2 -4.8  3.   3.1  5.2  0.3 11.6]]
#   0.0114

🖼 **Picture.** The same image through the network: 784 pixels squeezed to the 8 numbers in green, then expanded back to 784. The reconstruction is a slightly blurred 7.

In [ ]:
fig_ae_hourglass()

Nobody told the network that this is a 7, or what the 8 numbers should mean. **The rest of this notebook answers:** what makes it learn a useful code (§1), what it learns when the layers are linear (§2), how it's trained (§3), and what the codes look like (§4).

## 1. The idea: copy the input through a bottleneck

**The problem.** We have 60,000 images and no labels, and we want a short description of each image that keeps what matters. PCA does that with a linear projection. Can a network learn a better one?

**The idea.** Ask a network to output its own input. On its own that's trivial (copy each pixel through), so put a **bottleneck** in the middle: a layer of only $k$ units, far fewer than the 784 pixels. To pass through it, the image has to be summarized in $k$ numbers, and to rebuild it from those, the network has to learn what images in this dataset have in common: strokes, shapes, the typical ways each digit is written. An everyday picture: describing a face to a sketch artist over the phone in ten words. You'd say what's distinctive, not "pixel 1 is white".

**The formula, piece by piece.**

$$
z = f_\theta(x), \qquad \hat x = g_\phi(z), \qquad L(\theta, \phi) = \frac{1}{n}\sum_{i=1}^{n} \frac{1}{d}\, \big\lVert x_i - g_\phi(f_\theta(x_i)) \big\rVert^2
$$

| Symbol | Shape | In words |
|---|---|---|
| $x$ | $(d,)$ = (784,) | one input image, flattened |
| $f_\theta$ |  | the encoder, with weights $\theta$: `Linear(784, 128) → ReLU → Linear(128, k)` |
| $z$ | $(k,)$ = (8,) | the code |
| $g_\phi$ |  | the decoder, with weights $\phi$: `Linear(k, 128) → ReLU → Linear(128, 784) → Sigmoid` |
| $\hat x$ | $(d,)$ | the reconstruction, pixel values in $(0, 1)$ |
| $L$ | one number | the mean squared error per pixel (`nn.MSELoss()`), averaged over the images |

**No labels anywhere:** the target is the input itself. That's called **self-supervised** learning. Binary cross-entropy per pixel is a common alternative to the squared error when pixels are in $[0, 1]$ (`03` uses it).

**Counting the parameters (an exam favourite).** Each `Linear(a, b)` has $a \cdot b + b$:

| Layer | Calculation | Parameters |
|---|---|---|
| encoder `Linear(784, 128)` | 784 · 128 + 128 | 100,480 |
| encoder `Linear(128, 8)` | 128 · 8 + 8 | 1,032 |
| decoder `Linear(8, 128)` | 8 · 128 + 128 | 1,152 |
| decoder `Linear(128, 784)` | 128 · 784 + 784 | 101,136 |
| total |  | **203,800** |

**Remarks.** (1) The decoder doesn't have to mirror the encoder, but it usually does. (2) If $k \ge d$ the network can learn to copy and nothing useful is learned; the bottleneck (or noise, §5) is what forces a summary. (3) Training only needs the images: an AE can be trained on every unlabelled image you have, then its encoder reused as a feature extractor, as in `01`.

**In one sentence:** an autoencoder is trained to output its input through a narrow code, so the code must keep what matters about the input.

## 2. A linear autoencoder learns PCA

**The question.** Take away the nonlinearities: encoder $z = E x$ with $E$ `(k, d)`, decoder $\hat x = D z$ with $D$ `(d, k)`, data centred, squared error. What does gradient descent find?

**Proposition.** For centred data $x_1, \dots, x_n$, every minimizer of $\sum_i \lVert x_i - D E x_i \rVert^2$ over $D$ `(d, k)` and $E$ `(k, d)` makes $DE$ the orthogonal projection onto the span of the top-$k$ eigenvectors of the covariance matrix: the same subspace as PCA with $k$ components. The codes $z = Ex$ are PCA's scores, up to an invertible $k \times k$ change of coordinates.

**What it says, in words.** A linear autoencoder can't beat PCA and, trained well, matches it: it finds the same $k$-dimensional plane through the data. It may describe positions in that plane with different (rotated, stretched) axes.

**Small example.** The picture below trains one with Adam on the 8×8 digits ($d = 64$, $k = 2$) and compares it with PCA.

🖼 **Picture.** Left: the linear AE's error falls to PCA's. Middle and right: PCA's scores and the AE's codes, the same cloud turned and stretched.

In [ ]:
fig_linear_ae_pca()

In [ ]:
curve, pca_err, codes, Z, cosines = linear_ae_vs_pca()       # figures.py: 3000 Adam steps on the centred 8x8 digits
print(round(curve[-1], 4), round(pca_err, 4), cosines.round(4))
# → 3.3553 3.3553 [1. 1.]

**The proof in plain words.** Two steps. (1) Whatever plane the decoder's columns span, the best you can do for each point is to replace it by the closest point of that plane, which is its orthogonal projection; so at the optimum, $DE$ is a projection onto some $k$-dimensional plane. (2) Among all planes, the one whose projections lose the least squared length is the one that keeps the most variance: PCA's top-$k$ eigenvectors, which is Week 4's proposition.

**The proof.** Let $Q$ `(d, k)` have orthonormal columns spanning the columns of $D$ (assume $D$ has rank $k$). For each $x_i$, $D E x_i$ is some point of that plane, and the closest point of the plane to $x_i$ is $QQ^\top x_i$, so

$$
\sum_i \lVert x_i - DEx_i\rVert^2 \;\ge\; \sum_i \lVert x_i - QQ^\top x_i\rVert^2 \;=\; \sum_i \lVert x_i\rVert^2 - \sum_i \lVert Q^\top x_i\rVert^2,
$$

with equality when $E = (D^\top D)^{-1} D^\top$, which makes $DE = QQ^\top$. (The equality on the right is Pythagoras: $x_i = QQ^\top x_i + (x_i - QQ^\top x_i)$, two orthogonal pieces.) The first sum doesn't depend on the weights, so minimizing the error means maximizing

$$
\sum_i \lVert Q^\top x_i \rVert^2 = \operatorname{tr}\!\Big(Q^\top \Big(\sum_i x_i x_i^\top\Big) Q\Big) = (n-1)\, \operatorname{tr}(Q^\top S\, Q)
$$

over matrices $Q$ with orthonormal columns, where $S$ is the covariance matrix. By Week 4's result (the top eigenvector maximizes $w^\top S w$, and the top $k$ together maximize the sum), the maximum is reached when the columns of $Q$ span the top-$k$ eigenvectors, which is PCA's subspace. Finally, any invertible $A$ `(k, k)` gives $D A^{-1}$ and $A E$ with the same product $DE$, so the codes are only determined up to such an $A$. $\blacksquare$

**Remarks.** (1) **Centred data** is needed so the plane passes through the mean; with biases in the layers, the AE learns the centring itself. (2) **Squared error** is needed: with another loss the answer differs. (3) **Why the AE's axes aren't PCA's:** nothing in the loss prefers orthogonal, variance-ordered axes; that's PCA's extra convention. (4) **So why use networks at all?** Add nonlinearities and the code can describe curved structure that no plane can: the next figure shows the nonlinear AE beating PCA at the same $k$.

**In one sentence:** a linear autoencoder with squared error finds PCA's subspace; nonlinear layers are what let an autoencoder do better than PCA.

## 3. Training it

The training loop is the usual five lines; the only change is that the **target is the input**. `TensorDataset(X_tr)` holds only images, so each batch is a 1-tuple `(xb,)`. This is the exact code that made `ae8.pt` (about 5 seconds on the CPU):

In [ ]:
class AE(nn.Module):                            # exercise 2 asks you for this class from memory
    def __init__(self, d, k):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, k))
        self.decoder = nn.Sequential(nn.Linear(k, 128), nn.ReLU(), nn.Linear(128, d), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x))    # x -> z -> x_hat

torch.manual_seed(0)
ae = AE(784, 8)
loader = DataLoader(TensorDataset(X_tr), batch_size=128, shuffle=True)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
for epoch in range(5):
    ae.train()
    for (xb,) in loader:                        # each batch: a 1-tuple of (128, 784) images
        loss = loss_fn(ae(xb), xb)              # the target is the input itself
        opt.zero_grad(); loss.backward(); opt.step()
ae.eval()
with torch.no_grad():
    print(round(loss_fn(ae(X_te), X_te).item(), 4))    # → 0.0235   mean squared error per pixel, test set

**What the bottleneck size does.** Too small and the code can't hold enough: blurry, generic digits. Too large and the AE gets close to copying, so the code is less of a summary. The figure trains the same network with $k = 2, 8, 32$ and compares with PCA at $k = 8$:

🖼 **Picture.** Rows: the input, then reconstructions. With 2 numbers many digits come back as a generic 9 or 8; with 8 most are right; with 32 they're nearly copies. PCA with 8 components is much blurrier than the AE with 8.

In [ ]:
fig_bottleneck_sizes()

| Model | Code size | Test error (8 images above) |
|---|---|---|
| autoencoder | 2 | 0.0397 |
| autoencoder | 8 | 0.0198 |
| PCA | 8 | 0.0335 |
| autoencoder | 32 | 0.0083 |

**In one sentence:** train with the input as the target; choose $k$ small enough to force a summary and large enough to keep what you need.

## 4. The code space

With $k = 2$ every image becomes a point in a plane, which we can draw. `ae2.pt` is §3's network with $k = 2$. Colour each test image's code by its true digit, which the AE never saw:

🖼 **Picture.** Left: the codes of 5000 test images. Right: what the decoder makes of three chosen codes.

In [ ]:
fig_latent_ae2()

**What to see.** (1) The digits form groups, although no labels were used: similar images get similar codes. That's why encoders make good feature extractors. (2) The groups have odd shapes and sizes (the 1s spread from about $z_1 = 3$ to $18$) and there are **gaps**. (3) Decoding a point inside a group gives a good digit (A); between groups, a smudge (B); in empty space, a blob (C). The AE was only asked to rebuild training images, so it never learned what to do with codes no training image produced.

**Why that matters.** It means we can't **generate** new images by picking a random code: we wouldn't know where the valid codes are, and most picks land in a gap. `03`'s variational autoencoder fixes exactly this.

**A number for "good features".** Use the 8-number codes of `ae8.pt` as features for logistic regression, trained on 1,000 labelled images:

In [ ]:
from sklearn.linear_model import LogisticRegression
with torch.no_grad():
    Z_tr, Z_te = model.encoder(X_tr[:1000]).numpy(), model.encoder(X_te).numpy()    # (1000, 8), (10000, 8)
clf = LogisticRegression(max_iter=5000).fit(Z_tr, y_tr[:1000].numpy())
print(round(clf.score(Z_te, y_te.numpy()), 3))     # → 0.794   from 8 numbers per image, learned without labels

## 5. Denoising autoencoders

**The problem.** A big bottleneck rebuilds well but may just copy (with $k = 32$ the error was tiny). Is there another way to stop copying?

**The idea.** Corrupt the input, but keep the clean image as the target. Copying is now useless: the network has to know what a clean digit looks like to undo the noise. Same network, same loss; only the input changes:

$$
L = \frac{1}{n}\sum_{i} \frac{1}{d}\,\big\lVert x_i - g\big(f(\tilde x_i)\big)\big\rVert^2, \qquad \tilde x_i = \operatorname{clip}\big(x_i + \sigma\varepsilon_i,\; 0,\; 1\big), \quad \varepsilon_i \sim \mathcal{N}(0, I)
$$

$\tilde x_i$ is the noisy version (noise of standard deviation $\sigma$, then clipped back into $[0, 1]$); the loss compares the output with the **clean** $x_i$. In code, one line changes in the loop (the recipe of `figures.py`'s `train_dae`, which made `dae32.pt`):

In [ ]:
xb = X_tr[:4]
torch.manual_seed(0)
noisy = (xb + 0.5 * torch.randn_like(xb)).clamp(0, 1)   # the input: noise added, kept in [0, 1]
print(noisy.shape, round(nn.functional.mse_loss(noisy, xb).item(), 3))
# → torch.Size([4, 784]) 0.116
# in the loop:  loss = loss_fn(model(noisy), xb)        # noisy in, CLEAN target

🖼 **Picture.** A denoising AE ($k = 32$) on test digits it never saw. Top: the input; middle: its output; bottom: the clean original.

In [ ]:
fig_denoise()

**Remarks.** (1) Other corruptions work too: masking random pixels to 0 is the most common. (2) Masking big patches of an image and rebuilding them is exactly **MAE** (masked autoencoders, He et al. 2022), one of the main ways large vision models are pretrained today; masking words and predicting them is how BERT is pretrained (Week 8–9). (3) A denoising AE also learns where images "should" be, which is useful for anomaly detection (`04`).

## 6. Convolutional autoencoders and the transposed convolution

For images, the encoder is a CNN (Week 6): convolutions with stride 2 shrink 28 → 14 → 7. The decoder must grow 7 → 14 → 28, which needs a layer that **increases** the height and width: the **transposed convolution** (`nn.ConvTranspose2d`).

**What it does.** A convolution reads a window of the input and writes one number. A transposed convolution does the reverse: it reads **one** input number and writes a whole window into the output, the kernel scaled by that number. Where windows overlap, the values add up. With stride 2, the windows of neighbouring inputs start 2 pixels apart, so the output is about twice as big.

🖼 **Picture.** Left: kernel 2, stride 2: each input value stamps its own 2×2 block (no overlap), 2×2 → 4×4. Right: kernel 3, stride 1: the stamps overlap and add, 2×2 → 4×4.

In [ ]:
fig_conv_transpose()

**The size formula.** A convolution maps $n_{\text{in}}$ to $n_{\text{out}} = \lfloor (n_{\text{in}} + 2p - k)/s \rfloor + 1$ (Week 6). The transposed convolution with the same $k, p, s$ goes the other way; solving for $n_{\text{in}}$:

$$
n_{\text{out}} = (n_{\text{in}} - 1)\, s \;-\; 2p \;+\; k \;+\; \text{output\_padding}
$$

**Where it comes from, in words:** the first stamp covers $k$ pixels; each of the other $n_{\text{in}} - 1$ inputs shifts the next stamp by $s$; padding $p$ crops $p$ pixels from each side. `output_padding` adds extra rows at the end: needed because the convolution's $\lfloor \cdot \rfloor$ maps two sizes to the same output (14 and 13 both become 7 with $k = 3$, $s = 2$, $p = 1$), so the reverse has to be told which one you want. **Example:** $n = 7$, $k = 3$, $s = 2$, $p = 1$, output padding 1: $(7 - 1) \cdot 2 - 2 + 3 + 1 = 14$.

In [ ]:
conv_ae = nn.Sequential(
    nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),     # (1, 28, 28) -> (16, 14, 14)
    nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),    # -> (32, 7, 7): the code is 32*7*7 numbers
    nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),   # -> (16, 14, 14)
    nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid()) # -> (1, 28, 28)
x = torch.zeros(2, 1, 28, 28)
for layer in conv_ae:
    x = layer(x)
    if not isinstance(layer, (nn.ReLU, nn.Sigmoid)):
        print(tuple(x.shape))
# → (2, 16, 14, 14)
#   (2, 32, 7, 7)
#   (2, 16, 14, 14)
#   (2, 1, 28, 28)

**In one sentence:** a transposed convolution stamps a scaled kernel for every input value, so with stride $s$ it grows the image to $(n - 1)s - 2p + k$ (plus output padding); it's the decoder's mirror of a strided convolution.

## 7. Exercises

**Exercise 2 · The autoencoder as a class.** `Autoencoder(d, k)`: `self.encoder = nn.Sequential(Linear(d, 128), ReLU, Linear(128, k))` and `self.decoder = nn.Sequential(Linear(k, 128), ReLU, Linear(128, d), Sigmoid)`, created in this order; `forward(x)` returns the reconstruction.

**Example**: what goes in, and what should come out:
```python
Autoencoder(784, 8)(torch.zeros(3, 784)).shape     # → torch.Size([3, 784])
sum(p.numel() for p in Autoencoder(784, 8).parameters())   # → 203800
```

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, d, k):
        pass  # your code

    def forward(self, x):
        pass  # your code

In [ ]:
check(2)

**Exercise 3 · Reconstruction error per image.** For a model and a batch `X` `(N, d)`, return the mean squared error of **each** image: a tensor `(N,)`, the mean over the pixels (dim 1). Use `torch.no_grad()`. (`04` uses this as the anomaly score.)

**Example**: what goes in, and what should come out:
```python
recon_errors(nn.Identity(), torch.rand(5, 784))     # → tensor([0., 0., 0., 0., 0.])   a perfect copier
recon_errors(model, X_te[:3])                       # → tensor([0.0114, 0.0363, 0.0040])   (the ae8 model of §E)
```

In [ ]:
def recon_errors(model, X):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · One denoising training step.** Make `noisy = (xb + std * torch.randn_like(xb)).clamp(0, 1)`, compute the MSE between `model(noisy)` and the **clean** `xb`, then zero the gradients, backward, step. Return the loss as a Python float.

**Example**: what goes in, and what should come out:
```python
denoising_step(ae, opt, X_tr[:128], 0.5)     # → a float, e.g. about 0.06 early in training (the noise is random, so it varies)
```

In [ ]:
def denoising_step(model, opt, xb, std):
    pass  # your code

In [ ]:
check(4)

**Exercise 5 · The transposed-convolution size.** Return $(n - 1)s - 2p + k + \text{output\_padding}$.

**Example**: what goes in, and what should come out:
```python
deconv_out_size(7, 3, p=1, s=2, output_padding=1)    # → 14
deconv_out_size(2, 2, s=2)                          # → 4
```

In [ ]:
def deconv_out_size(n, k, p=0, s=1, output_padding=0):
    pass  # your code

In [ ]:
check(5)

**Recipe (autoencoder).** Encoder to a $k$-dimensional code, mirrored decoder, sigmoid output for pixels in $[0, 1]$; MSE (or BCE) between the output and the input; the usual loop with `(xb,)` batches. Pick $k$ by the reconstruction error you can accept; add noise to the input (denoising) to learn more robust codes; for images use strided convolutions down and transposed convolutions up.

In [ ]:
for n in range(1, 6):
    check(n)

---
Solutions: `solutions/02_solutions.py`. Next: `03_vae.ipynb`.